# Codex CLI 0.167.0에서 달라진 점을 정리하고, codex agents 명령 센터로 여러 세션을 관리하는 법과 Vertex AI 설정 마법사 쓰는 법을 쓴다

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `OPENAI_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['OPENAI_API_KEY']:
    os.environ[name] = userdata.get(name)

### 0.167.0 설치와 버전·옵션 확인

In [ ]:
%%bash
npm i -g @openai/codex@0.167.0
codex --version
codex agents --help | head -30

### Codex 설치·로그인과 예제 저장소·세션 만들기

In [ ]:
%%bash
npm i -g @openai/codex@0.167.0
codex --version
printenv OPENAI_API_KEY | codex login --with-api-key
mkdir -p /work/demo && cd /work/demo && git init -q
printf 'def add(a, b):\n    return a + b\n' > calc.py
echo '# demo' > README.md
git add -A && git -c user.email=a@b.c -c user.name=t commit -qm init
codex exec -s danger-full-access 'calc.py를 읽고 한 문장으로 설명해 줘'
codex exec -s danger-full-access 'README.md 첫 줄을 읽고 알려 줘'

### codex agents 옵션 확인

In [ ]:
%%bash
codex agents --help | grep -A1 -e '--no-daemon'

### 가상 터미널 도우미 만들기 (TUI 화면 캡처용)

In [ ]:
%%bash
mkdir -p /work/lib /work/outputs /work/demo
pip install -q pexpect pyte pillow
cat > /work/lib/term.py <<'E'
import pexpect, pyte, time, os
from PIL import Image, ImageDraw, ImageFont
W,H=140,40
F=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf',16)
K=None
for p in ['/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']:
    if os.path.exists(p): K=ImageFont.truetype(p,16,index=6) if False else ImageFont.truetype(p,16)
class T:
    def __init__(s,cmd,cwd='/work/demo',env=None):
        e=dict(os.environ,TERM='xterm-256color',COLORTERM='truecolor'); e.update(env or {})
        s.sc=pyte.Screen(W,H); s.st=pyte.ByteStream(s.sc)
        s.p=pexpect.spawn('/bin/bash',['-lc',cmd],cwd=cwd,env=e,dimensions=(H,W))
    def pump(s,sec):
        end=time.time()+sec
        while time.time()<end:
            try: d=s.p.read_nonblocking(65536,0.2); s.st.feed(d)
            except pexpect.TIMEOUT: pass
            except pexpect.EOF: break
    def send(s,x,sec=2): s.p.send(x); s.pump(sec)
    def text(s): return "\n".join("".join((s.sc.buffer[y][c].data or "") for c in range(W)).rstrip() for y in range(H))
    def png(s,path):
        cw,ch=10,22
        im=Image.new('RGB',(W*cw+20,H*ch+20),(18,18,24)); d=ImageDraw.Draw(im)
        cols={'default':None,'black':(0,0,0),'red':(230,80,80),'green':(90,210,110),'brown':(230,190,80),'blue':(90,140,240),'magenta':(210,110,220),'cyan':(80,210,220),'white':(230,230,230),'brightblack':(130,130,140)}
        for y in range(H):
            x=0
            for c in range(W):
                ch_=s.sc.buffer[y][c]
                if ch_.data=='' : continue
                fg=ch_.fg; col=cols.get(fg) or ((int(fg[0:2],16),int(fg[2:4],16),int(fg[4:6],16)) if len(fg)==6 else (225,225,225))
                bg=ch_.bg
                if bg!='default' and len(bg)==6: d.rectangle([10+c*cw,10+y*ch,10+(c+1)*cw,10+(y+1)*ch],fill=tuple(int(bg[i:i+2],16) for i in (0,2,4)))
                if ch_.reverse: d.rectangle([10+c*cw,10+y*ch,10+(c+1)*cw,10+(y+1)*ch],fill=(200,200,200)); col=(18,18,24)
                if ch_.data!=' ':
                    f=F if ord(ch_.data)<0x2e80 else (K or F)
                    d.text((10+c*cw,10+y*ch),ch_.data,font=f,fill=col)
        im.save(path)
E
echo ok

### 세션 두 개를 만들 조작 스크립트 작성

In [ ]:
%%bash
cat > /work/lib/g3.py <<'E'
import sys; sys.path.insert(0,'/work/lib')
from term import T
t=T('codex agents --no-daemon -c sandbox_mode=\\"danger-full-access\\" -c approval_policy=\\"never\\"')
t.pump(6)
t.send('n',3)
t.send('calc.py에 sub 함수를 추가하고 pytest 스타일 테스트 파일 test_calc.py도 만들어 줘',1)
t.send('\r',3); t.png('/work/outputs/agents_04_session_running.png')
t.send('\x1b[D',3); print(t.text()[:2500]); t.png('/work/outputs/agents_05_list_working.png')
t.pump(40); print('=====AFTER'); print(t.text()[:2500]); t.png('/work/outputs/agents_06_list_ready_unread.png')
open('/work/lib/state.txt','w').write(t.text())
E
cat > /work/lib/g3b.py <<'E'
import sys; sys.path.insert(0,'/work/lib')
from term import T, W
t=T('codex agents --no-daemon -c sandbox_mode=\\"danger-full-access\\" -c approval_policy=\\"never\\"')
def marks(tag):
    print('--',tag)
    for y in range(4,12):
        row=t.text().split('\n')[y]
        if '●' in row or '○' in row:
            c=row.index('●') if '●' in row else row.index('○')
            ch=t.sc.buffer[y][c]; print(y,repr(row[:50].strip()),'fg=',ch.fg,'bold=',ch.bold)
t.pump(6); marks('start')
t.send('n',3)
t.send('README.md에 사용법 섹션을 두 줄로 추가해 줘',1)
t.send('\r',2); t.send('\x1b[D',2)
t.pump(30); print(t.text()[:1800]); marks('after2'); t.png('/work/outputs/agents_07_unread.png')
t.send('\x1b[B',1); t.send('\x1b[B',1)
t.send('\r',4); t.png('/work/outputs/agents_08_opened_transcript.png')
t.send('\x1b[D',3); marks('after_open'); print(t.text()[:1500]); t.png('/work/outputs/agents_09_after_read.png')
E
echo ok

### 첫 세션 만들고 Working → Ready 전환 확인

In [ ]:
%%bash
cd /work && python3 -I lib/g3.py

### 두 번째 세션으로 최근순 정렬·읽지 않음 표시 확인

In [ ]:
%%bash
cd /work && python3 -I lib/g3b.py

### touch 명령에 승인을 요구하는 규칙과 조작 스크립트 만들기

In [ ]:
%%bash
mkdir -p ~/.codex/rules
cat > ~/.codex/rules/default.rules <<'E'
prefix_rule(pattern=["touch"], decision="prompt")
E
cat > /work/lib/g4b.py <<'E'
import sys; sys.path.insert(0,'/work/lib')
from term import T
t=T('codex agents --no-daemon -c sandbox_mode=\\"danger-full-access\\" -c approval_policy=\\"on-request\\"')
t.pump(6)
t.send('n',3)
t.send('서브에이전트 2개를 띄워서, 각각 /work/demo 안에서 `touch a.txt` 와 `touch b.txt` 셸 명령을 실행하게 해 줘. 직접 실행하지 말고 반드시 서브에이전트에게 시켜.',1)
t.send('\r',30); t.png('/work/outputs/agents_10_approval_modal.png')
t.send('o',3); print(t.text()[-2500:]); t.png('/work/outputs/agents_12_after_o.png')
t.send('\x1b',3); print('=====ESC'); print(t.text()[-1800:]); t.png('/work/outputs/agents_13_after_esc.png')
t.send('/subagents',1); t.send('\r',3); print('=====PICKER'); print(t.text()[-2500:]); t.png('/work/outputs/agents_14_subagent_picker.png')
E
rm -f /work/demo/a.txt /work/demo/b.txt
echo ok

### 서브에이전트 승인 요청이 선택기 위쪽에 뜨는지 확인

In [ ]:
%%bash
cd /work && python3 -I lib/g4b.py

### Vertex AI 설정 마법사 조작 스크립트 만들기

In [ ]:
%%bash
mkdir -p /tmp/ch5
cat > /work/lib/g5.py <<'E'
import sys; sys.path.insert(0,'/work/lib')
from term import T
t=T('codex --enable gcp_setup_wizard',env={'CODEX_HOME':'/tmp/ch5','OPENAI_API_KEY':''})
t.pump(8); print(t.text()); t.png('/work/outputs/vertex_01_login.png')
t.send('3',2); print('=====SUB'); print(t.text()); t.png('/work/outputs/vertex_02_sign_in_another_way.png')
t.send('3',3); print('=====VTX'); print(t.text()); t.png('/work/outputs/vertex_03_wizard_project.png')
t.send('my-project-id',1); t.send('\r',3); print('=====NEXT'); print(t.text()); t.png('/work/outputs/vertex_04_wizard_location.png')
t.send('\r',2); print('=====NEXT2'); print(t.text()); t.png('/work/outputs/vertex_05_wizard_credentials.png')
t.send('\r',6); print('=====NEXT3'); print(t.text()); t.png('/work/outputs/vertex_06_wizard_result.png')
E
echo ok

### gcp_setup_wizard 켜고 Sign in another way → Vertex AI 설정 진행

In [ ]:
%%bash
cd /work && python3 -I lib/g5.py | grep -v '^$'